In [1]:
import pandas as pd
import duckdb
import os
# Create output directory if it doesn't exist
output_dir = '../../../data/woah/output'
os.makedirs(output_dir, exist_ok=True)

file_url='https://oss.resilientservice.mooo.com/resilentpublic/pathogens/wahis/raw/infur_20251103.parquet'

In [2]:
db1=duckdb.sql(f"SELECT * FROM read_parquet('{file_url}') ")

In [3]:
diseases_df = duckdb.sql("""
                    SELECT DISTINCT disease_id,
                                    reporting_level,
                                    strain_eng,
                                    sero_sub_genotype_eng,
                                    disease_eng
                    FROM db1
                    """).df()

diseases_df

,disease_id,reporting_level,strain_eng,sero_sub_genotype_eng,disease_eng
0,786,disease,None,None,Newcastle disease virus (Inf. with)
1,677,serotype/subtype/genotype,None,Indiana,Vesicular stomatitis (-2014)
2,62,serotype/subtype/genotype,None,Asia 1,Foot and mouth disease virus (Inf. with)
3,83,disease,None,None,Botulism (-2014)
4,870,disease,None,None,Varroa spp. (Inf. of honey bees with) (Varroosis)
...,...,...,...,...,...
282,301,strain,'-,H5N3,Influenza A viruses of high pathogenicity (Inf...
283,297,strain,'-,H5N2,Influenza A viruses of high pathogenicity (Inf...
284,375,strain,'-,H7N9,High pathogenicity avian influenza viruses (po...
285,139,strain,'-,1,African horse sickness virus (Inf. with)


In [20]:
diseases_df = duckdb.sql("""
                    SELECT DISTINCT disease_id,
                                    reporting_level,
                                    strain_eng,
                                    sero_sub_genotype_eng,
                                    disease_eng
                    FROM db1
                    """).df()

diseases_df

,disease_id,reporting_level,strain_eng,sero_sub_genotype_eng,disease_eng
0,60,disease,None,None,Anthrax
1,631,serotype/subtype/genotype,None,H7N7,High pathogenicity avian influenza viruses (po...
2,557,serotype/subtype/genotype,None,H5N1,High pathogenicity avian influenza viruses (po...
3,409,disease,None,None,Equine encephalomyelitis (Eastern and Western)...
4,823,disease,None,None,Q fever
...,...,...,...,...,...
282,158,strain,'-,4,Bluetongue virus (Inf. with)
283,313,strain,'-,H5N6,Influenza A viruses of high pathogenicity (Inf...
284,883,strain,clade 2.3.4.4.b - H5N2 (HPAI),H5N2,High pathogenicity avian influenza viruses (po...
285,113,disease,None,None,Crimean Congo haemorrhagic fever (2006-)


In [5]:
duckdb.sql("""
                    SELECT DISTINCT disease_id,
                                    reporting_level,
                                    strain_eng,
                                    sero_sub_genotype_eng,
                                    disease_eng
                    FROM db1
                        WHERE disease_eng LIKE 'New world screwworm%'
                    """).df()



,disease_id,reporting_level,strain_eng,sero_sub_genotype_eng,disease_eng
0,792,disease,None,None,New world screwworm (Cochliomyia hominivorax)


In [6]:
nws_df=duckdb.sql("""
           SELECT disease_id,
                  reporting_level,
                  strain_eng,
                  sero_sub_genotype_eng,
                  disease_eng,
                  country,
                  region,
                  "reason of notification",
                  Species,
                  quantitative_unit,
                  MIN(Outbreak_start_date)          as Outbreak_start_date,
                  MAX(Outbreak_end_date)            as Outbreak_end_date,
                  SUM(COALESCE(susceptible, 0))     as susceptible,
                  SUM(COALESCE(cases, 0))           as cases,
                  SUM(COALESCE(dead, 0))            as dead,
                  SUM(COALESCE(killed_disposed, 0)) as killed_disposed,
                  SUM(COALESCE(slaughtered, 0))     as slaughtered,
                  SUM(COALESCE(vaccinated, 0))      as vaccinated,
                  SUM(COALESCE(morbidity, 0))       as morbidity,
                  SUM(COALESCE(mortality, 0))       as mortality
           FROM db1
           WHERE disease_eng LIKE 'New world screwworm%'
           GROUP BY epi_event_id,
                    disease_id,
                    reporting_level,
                    strain_eng,
                    sero_sub_genotype_eng,
                    disease_eng,
                    country,
                    region,
                    "reason of notification",
                    Species,
                    quantitative_unit
           """).df()
nws_df

,disease_id,reporting_level,strain_eng,sero_sub_genotype_eng,disease_eng,country,region,reason of notification,Species,quantitative_unit,Outbreak_start_date,Outbreak_end_date,susceptible,cases,dead,killed_disposed,slaughtered,vaccinated,morbidity,mortality
0,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Nicaragua,Americas,reccurence disease,Swine,Animal,2024-03-26,2025-02-06,32134.0,1222.0,1.0,0.0,0.0,0.0,0.0,0.0
1,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Mexico,Americas,First occ in zone,Equidae (dom),Animal,2025-02-21,2025-07-25,142.0,54.0,0.0,0.0,0.0,0.0,0.0,0.0
2,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Mexico,Americas,First occ in zone,Swine,Animal,2025-03-13,2025-07-23,145.0,27.0,0.0,0.0,0.0,0.0,0.0,0.0
3,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Mexico,Americas,First occ in zone,Sheep,Animal,2025-04-28,2025-07-13,805.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0
4,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),United States of America,Americas,reccurence disease,Dogs,Animal,2016-07-13,2017-03-23,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Guatemala,Americas,reccurence disease,Equidae (dom),Animal,2025-06-03,NaT,306.0,55.0,0.0,0.0,0.0,0.0,0.0,0.0
129,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Guatemala,Americas,reccurence disease,Swine,Animal,2025-06-03,NaT,348.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0
130,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Guatemala,Americas,reccurence disease,Goats,Animal,2025-06-17,NaT,44.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0
131,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Belize,Americas,First occ in zone,Equidae (dom),Animal,2025-05-29,2025-07-11,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0


# Task Summary

Based on the user request area being empty, I cannot determine what specific task you'd like me to perform on the `wahis_outbreaks.ipynb` notebook.

However, I can see that the notebook currently:
1. Loads WAHIS outbreak data from a parquet file
2. Queries distinct diseases
3. Filters for "New world screwworm" outbreaks with aggregated statistics

**Please provide your specific request.** For example:
- Add data visualization/plotting
- Export results to a file
- Perform additional analysis or filtering
- Create summary statistics
- Add data quality checks
- Transform or reshape the data

Once you clarify what you'd like to do, I'll provide the appropriate code cells for the notebook.


In [24]:
outbreaks_last_180_df = duckdb.sql("""
                                   SELECT disease_id,
                                          reporting_level,
                                          strain_eng,
                                          sero_sub_genotype_eng,
                                          disease_eng,
                                          country,
                                          region,
                                          "reason of notification",
                                          Species,
                                          quantitative_unit,
                                       ANY_VALUE(Reporting_date),
                                          MIN(Outbreak_start_date)          as Outbreak_start_date,
                                          MAX(Outbreak_end_date)            as Outbreak_end_date,
                                          SUM(COALESCE(susceptible, 0))     as susceptible,
                                          SUM(COALESCE(cases, 0))           as cases,
                                          SUM(COALESCE(dead, 0))            as dead,
                                          SUM(COALESCE(killed_disposed, 0)) as killed_disposed,
                                          SUM(COALESCE(slaughtered, 0))     as slaughtered,
                                          SUM(COALESCE(vaccinated, 0))      as vaccinated,
                                          SUM(COALESCE(morbidity, 0))       as morbidity,
                                          SUM(COALESCE(mortality, 0))       as mortality
                                   FROM db1
                                   WHERE disease_eng LIKE 'New world screwworm%'
                                     AND Reporting_date >= CURRENT_DATE - INTERVAL 365 DAY
                                   GROUP BY epi_event_id,
                                       disease_id,
                                       reporting_level,
                                       strain_eng,
                                       sero_sub_genotype_eng,
                                       disease_eng,
                                       country,
                                       region,
                                       "reason of notification",
                                       Species,
                                       quantitative_unit,
                                       Reporting_date,
                                       QUALIFY ROW_NUMBER() OVER (PARTITION BY epi_event_id ORDER BY Reporting_date
                                       DESC) = 1
                                   """).df()

outbreaks_last_180_df


,disease_id,reporting_level,strain_eng,sero_sub_genotype_eng,disease_eng,country,region,reason of notification,Species,quantitative_unit,...,Outbreak_start_date,Outbreak_end_date,susceptible,cases,dead,killed_disposed,slaughtered,vaccinated,morbidity,mortality
0,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Belize,Americas,reccurence disease,Dogs,Animal,...,2025-07-03,NaT,113.0,33.0,0.0,0.0,0.0,0.0,0.0,0.0
1,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Honduras,Americas,reccurence disease,Dogs,Animal,...,2024-10-25,2025-03-14,2691.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0
2,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Mexico,Americas,First occ in zone,Cattle,Animal,...,2025-06-04,2025-07-24,573.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0
3,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Mexico,Americas,First occ in zone,Cattle,Animal,...,2025-10-12,NaT,20.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Mexico,Americas,First occ in zone,Swine,Animal,...,2025-07-07,2025-07-22,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
5,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Mexico,Americas,First occ in zone,Cattle,Animal,...,2025-10-08,NaT,67.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
6,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Mexico,Americas,First occ in zone,Cattle,Animal,...,2025-06-28,2025-07-21,451.0,11.0,0.0,0.0,0.0,0.0,0.0,0.0
7,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Honduras,Americas,First occ in zone,Dogs,Animal,...,2025-05-16,NaT,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
8,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Belize,Americas,First occ in zone,Dogs,Animal,...,2025-07-09,NaT,41.0,14.0,0.0,0.0,0.0,0.0,0.0,0.0
9,792,disease,None,None,New world screwworm (Cochliomyia hominivorax),Mexico,Americas,reccurence disease,Dogs,Animal,...,2025-04-22,2025-05-07,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [48]:
outbreaks_df=duckdb.sql("""
           SELECT epi_event_id,
               Report_id,
                  disease_id,
                  reporting_level,
                  strain_eng,
                  sero_sub_genotype_eng,
                  disease_eng,
                  country,
                  region,
                  "reason of notification",
                  Species,
                  quantitative_unit,
                  MIN(Outbreak_start_date)          as Outbreak_start_date,
                  MAX(Outbreak_end_date)            as Outbreak_end_date,
                  SUM(COALESCE(susceptible, 0))     as susceptible,
                  SUM(COALESCE(cases, 0))           as cases,
                  SUM(COALESCE(dead, 0))            as dead,
                  SUM(COALESCE(killed_disposed, 0)) as killed_disposed,
                  SUM(COALESCE(slaughtered, 0))     as slaughtered,
                  SUM(COALESCE(vaccinated, 0))      as vaccinated,
                  SUM(COALESCE(morbidity, 0))       as morbidity,
                  SUM(COALESCE(mortality, 0))       as mortality
           FROM db1

           GROUP BY Report_id, epi_event_id,
                    disease_id,
                    reporting_level,
                    strain_eng,
                    sero_sub_genotype_eng,
                    disease_eng,
                    country,
                    region,
                    "reason of notification",
                    Species,
                    quantitative_unit
           """).df()
outbreaks_df

,epi_event_id,Report_id,disease_id,reporting_level,strain_eng,sero_sub_genotype_eng,disease_eng,country,region,reason of notification,...,Outbreak_start_date,Outbreak_end_date,susceptible,cases,dead,killed_disposed,slaughtered,vaccinated,morbidity,mortality
0,296,1026,677,serotype/subtype/genotype,None,Indiana,Vesicular stomatitis (-2014),Bolivia,Americas,reccurence disease,...,2005-02-10,2005-05-04,1000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,353,1269,839,disease,None,None,Scrapie,Slovenia,Europe,reccurence disease,...,2005-03-09,2005-03-25,615.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
2,298,1241,789,serotype/subtype/genotype,None,New Jersey,Vesicular stomatitis (-2014),United States of America,Americas,reccurence disease,...,2005-09-08,2006-01-30,805.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0
3,211,1217,59,disease,None,None,Paenibacillus larvae (Inf. of honey bees with)...,Chile,Americas,First occ in zone,...,2006-01-05,2006-03-28,385.0,22.0,0.0,22.0,0.0,0.0,0.0,0.0
4,75,197,553,serotype/subtype/genotype,None,H5,High pathogenicity avian influenza viruses (po...,Sudan,Africa,First occ in zone,...,2006-04-01,2006-09-04,65400.0,63400.0,63400.0,2000.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39709,4070,172919,293,strain,'-,H5N1,Influenza A viruses of high pathogenicity (Inf...,Hungary,Europe,reccurence disease,...,2025-03-06,2025-03-07,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
39710,6662,177401,172,strain,'-,8,Bluetongue virus (Inf. with),Slovenia,Europe,New strain in countr,...,2025-07-30,NaT,254.0,18.0,7.0,0.0,0.0,0.0,0.0,0.0
39711,6945,177470,684,disease,None,None,Batrachochytrium dendrobatidis (Inf. with)(2009-),Singapore,Asia,reccurence disease,...,2024-12-19,2024-12-31,0.0,61.0,21.0,1149.0,0.0,0.0,0.0,0.0
39712,6772,177508,842,disease,None,None,Aethina tumida (Inf. with)(Small hive beetle)(...,Italy,Europe,reccurence disease,...,2025-10-09,NaT,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:



# Get unique diseases and create a separate file for each
unique_diseases = outbreaks_df['disease_eng'].unique()

for disease in unique_diseases:
    # Filter outbreaks for this disease
    disease_outbreaks = outbreaks_df[outbreaks_df['disease_eng'] == disease]

    # Create a safe filename from disease name
    safe_filename = disease.replace('/', '_').replace(' ', '_').lower()
    output_path = os.path.join(output_dir, f'{safe_filename}_outbreaks.parquet')

    # Write to parquet file
    disease_outbreaks.to_parquet(output_path)
    print(f"Disease: {disease} | Records: {len(disease_outbreaks)} | File: {output_path}")

print(f"\nTotal unique diseases: {len(unique_diseases)}")


Disease: Foot and mouth disease virus (Inf. with) | Records: 879 | File: ../../../data/woah/output/foot_and_mouth_disease_virus_(inf._with)_outbreaks.parquet
Disease: Scrapie | Records: 41 | File: ../../../data/woah/output/scrapie_outbreaks.parquet
Disease: Enzootic bovine leukosis | Records: 6 | File: ../../../data/woah/output/enzootic_bovine_leukosis_outbreaks.parquet
Disease: High pathogenicity avian influenza viruses (poultry) (Inf. with) | Records: 1629 | File: ../../../data/woah/output/high_pathogenicity_avian_influenza_viruses_(poultry)_(inf._with)_outbreaks.parquet
Disease: Bonamia ostreae (Inf. with) | Records: 10 | File: ../../../data/woah/output/bonamia_ostreae_(inf._with)_outbreaks.parquet
Disease: Vesicular stomatitis (-2014) | Records: 39 | File: ../../../data/woah/output/vesicular_stomatitis_(-2014)_outbreaks.parquet
Disease: Peste des petits ruminants virus (Inf. with) | Records: 134 | File: ../../../data/woah/output/peste_des_petits_ruminants_virus_(inf._with)_outbreak

In [11]:
output_path = os.path.join(output_dir, f'outbreaks_summaries.parquet')
outbreaks_df.to_parquet(output_path)

In [13]:
output_path = os.path.join(output_dir, f'outbreaks_summaries.csv')
outbreaks_df.to_csv(output_path)

In [49]:

for disease_id in diseases_df['disease_id']:
    name= diseases_df[diseases_df['disease_id']==disease_id]['disease_eng'].unique()
    if len(name)==0:
        print(f'bad id {disease_id}')
    else:
        print(f'disease_id  {disease_id} {len(name)}')
        print (f'name: {name[0]}')
    # Filter outbreaks for this disease
    disease_outbreaks = duckdb.sql(f"""
           SELECT Report_id,Outbreak_id,
                  disease_id,
                  reporting_level,
                  strain_eng,
                  sero_sub_genotype_eng,
                  disease_eng,
                  country,
                  region,
                  "reason of notification",
                  Species,
                  quantitative_unit,
                  Outbreak_start_date        as Outbreak_start_date,
                  Outbreak_end_date           as Outbreak_end_date,
                  susceptible    as susceptible,
                  cases           as cases,
                  dead           as dead,
                  killed_disposed as killed_disposed,
                  slaughtered     as slaughtered,
                  vaccinated     as vaccinated,
                  morbidity      as morbidity,
                  mortality      as mortality
                  FROM db1
                  WHERE disease_id = {disease_id}
               """).df()

    # Create a safe filename from disease name
    safe_filename = name[0].replace('/', '_').replace(' ', '_').lower()
    output_path = os.path.join(output_dir, f'{safe_filename}_outbreaks.parquet')

    # Write to parquet file
    disease_outbreaks.to_parquet(output_path)
    print(f"Disease: {name} | Records: {len(disease_outbreaks)} | File: {output_path}")

print(f"\nTotal unique diseases: {len(unique_diseases)}")

disease_id  60 1
name: Anthrax
Disease: 60 | Records: 733 | File: ../../../data/woah/output/anthrax_outbreaks.parquet
disease_id  631 1
name: High pathogenicity avian influenza viruses (poultry) (Inf. with)
Disease: 60 | Records: 16 | File: ../../../data/woah/output/high_pathogenicity_avian_influenza_viruses_(poultry)_(inf._with)_outbreaks.parquet
disease_id  557 1
name: High pathogenicity avian influenza viruses (poultry) (Inf. with)
Disease: 60 | Records: 7932 | File: ../../../data/woah/output/high_pathogenicity_avian_influenza_viruses_(poultry)_(inf._with)_outbreaks.parquet
disease_id  409 1
name: Equine encephalomyelitis (Eastern and Western)(-2005)
Disease: 60 | Records: 1 | File: ../../../data/woah/output/equine_encephalomyelitis_(eastern_and_western)(-2005)_outbreaks.parquet
disease_id  823 1
name: Q fever
Disease: 60 | Records: 31 | File: ../../../data/woah/output/q_fever_outbreaks.parquet
disease_id  67 1
name: Avian infectious bronchitis
Disease: 60 | Records: 12 | File: ../.